In [1]:
import psycopg as pg
import pandas as pd
import os
import mlflow
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from category_encoders import CatBoostEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from catboost import CatBoostClassifier

In [2]:
with open('.env', 'r', encoding='utf-8') as env_file:
    env_text = env_file.read()

env_vars = pd.DataFrame([[s.split('=')[0],s.split('=')[1]] for s in env_text.split('\n')], columns=['variable','value'])
postgres_vars = env_vars[env_vars['variable'].str.startswith('DB_DEST')]

pg_host = postgres_vars.value[postgres_vars.variable.str.endswith('HOST')].values[0]
pg_port = postgres_vars.value[postgres_vars.variable.str.endswith('PORT')].values[0]
pg_name = postgres_vars.value[postgres_vars.variable.str.endswith('NAME')].values[0]
pg_user = postgres_vars.value[postgres_vars.variable.str.endswith('USER')].values[0]
pg_pswd = postgres_vars.value[postgres_vars.variable.str.endswith('PASSWORD')].values[0]

In [3]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": pg_host, 
    "port": pg_port,
    "dbname": pg_name,
    "user": pg_user,
    "password": pg_pswd,
}
assert all([var_value != "" for var_value in list(postgres_credentials.values())])

connection.update(postgres_credentials)

# определим название таблицы, в которой хранятся наши данные.
TABLE_NAME = "clean_users_churn"

# эта конструкция создаёт контекстное управление для соединения с базой данных 
# оператор with гарантирует, что соединение будет корректно закрыто после выполнения всех операций 
# закрыто оно будет даже в случае ошибки, чтобы не допустить "утечку памяти"
with pg.connect(**connection) as conn:

# создаёт объект курсора для выполнения запросов к базе данных
# с помощью метода execute() выполняется SQL-запрос для выборки данных из таблицы TABLE_NAME
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
                
        # извлекаем все строки, полученные в результате выполнения запроса
        data = cur.fetchall()

        # получает список имён столбцов из объекта курсора
        columns = [col[0] for col in cur.description]

# создаёт объект DataFrame из полученных данных и имён столбцов. 
# это позволяет удобно работать с данными в Python, используя библиотеку Pandas.
df = pd.DataFrame(data, columns=columns)

In [ ]:
# определяем основные credentials, которые нужны для подключения к MLflow
# важно, что credentials мы передаём для себя как пользователей Tracking Service
# у вас должен быть доступ к бакету, в который вы будете складывать артефакты
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net" #endpoint бакета от YandexCloud
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID") # получаем id ключа бакета, к которому подключён MLFlow, из .env
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY") # получаем ключ бакета, к которому подключён MLFlow, из .env
# определяем глобальные переменные
# поднимаем MLflow локально
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

EXPERIMENT_NAME = "churn_vds"

In [8]:
with open('fitted_model.pkl', 'rb') as fd:
    model = joblib.load(fd) 

prediction = model.predict(df)
proba = model.predict_proba(df)

X_test = df
y_test = df['target']

In [9]:
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, precision_score, recall_score, log_loss

In [39]:
confusion_matrix(y_test, prediction, normalize='all').ravel()

array([1.91679682e-02, 7.15462161e-01, 7.09924748e-04, 2.64659946e-01])

In [40]:
metrics = {}

_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()
auc = roc_auc_score(y_test, proba[:,1])
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, prediction)

metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [ ]:
experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id
    
    # логируем метрики эксперимента
    # предполагается, что переменная stats содержит словарь с метриками,
    # объявлять переменную stats не надо,
    # где ключи — это названия метрик, а значения — числовые значения метрик
    mlflow.log_metrics(stats)
    
    # логируем файлы как артефакты эксперимента — 'columns.txt' и 'users_churn.csv'
    mlflow.log_artifact('columns.txt', "dataframe") 
    mlflow.log_artifact('users_churn.csv', "dataframe") 

In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
# получаем данные о запуске эксперимента по его уникальному идентификатору
run = mlflow.get_run(run_id)

In [ ]:
assert run.info.status=='FINISHED'